# TicketOps — training IndoBERT di Colab
Pilih **Runtime → Change runtime type → GPU**. Upload `ticketops-source.zip` yang disertakan dalam deliverable. Notebook ini belum dijalankan pada saat penyerahan; tidak ada skor IndoBERT yang diklaim. Data bootstrap bersifat sintetis dan belum ditinjau manusia. Training dan pemilihan memakai development; test hanya dievaluasi setelah konfigurasi dibekukan.

In [ ]:
from google.colab import files
import zipfile, pathlib, os, subprocess, sys, json
uploaded = files.upload()
archive = next(name for name in uploaded if name.endswith('.zip'))
destination = pathlib.Path('/content/ticketops-work')
with zipfile.ZipFile(archive) as z:
    for member in z.infolist():
        target = (destination / member.filename).resolve()
        if not target.is_relative_to(destination.resolve()):
            raise ValueError('Unsafe archive path')
    z.extractall(destination)
os.chdir(destination / 'ticketops')
print(pathlib.Path.cwd())

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', './backend[training]'], check=True)
import torch
assert torch.cuda.is_available(), 'Pilih runtime GPU atau lanjutkan saat kuota tersedia.'
print(torch.cuda.get_device_name(0))
subprocess.run([sys.executable, 'ml/build_dataset.py'], check=True)

## 1. Bandingkan learning rate pada development
Maksimal lima epoch, early stopping dua evaluasi, batch 16, panjang 256 token. Checkpoint disimpan setiap epoch. Akses GPU gratis tidak dijamin dan sesi dapat berakhir; simpan artefak berkala.

In [ ]:
for lr in ['2e-5', '3e-5']:
    subprocess.run([sys.executable, 'ml/train_indobert.py', '--learning-rate', lr, '--seed', '42'], check=True)
candidates = list(pathlib.Path('ml/artifacts').glob('indobert-*-seed42/selection.json'))
best = max(candidates, key=lambda p: json.loads(p.read_text())['development_macro_f1'])
selection = json.loads(best.read_text())
print('Konfigurasi terpilih di development:', selection)
pathlib.Path('ml/artifacts/frozen-selection.json').write_text(json.dumps({'candidate': str(best.parent), **selection}, indent=2))

## 2. Stabilitas dan evaluasi konfigurasi yang sudah dibekukan
Ulangi learning rate terpilih dengan seed 17 dan 71. Jangan mengganti konfigurasi berdasarkan test. Laporkan hasil tiap seed dan keragaman keluarga data. Semua hasil masih benchmark sintetis.

In [ ]:
for seed in [17, 71]:
    subprocess.run([sys.executable, 'ml/train_indobert.py', '--learning-rate', str(selection['learning_rate']), '--seed', str(seed)], check=True)
for seed in [42, 17, 71]:
    folder = pathlib.Path('ml/artifacts') / f"indobert-lr{selection['learning_rate']}-seed{seed}"
    subprocess.run([sys.executable, 'ml/evaluate_indobert.py', str(folder)], check=True)

## 3. Ekspor dan uji di laptop
Download kandidat seed42 terpilih beserta laporan. Ekstrak ke root proyek lokal, lalu jalankan `python ml/promote_model.py ml/artifacts/<folder-kandidat>`. Gate membandingkan development dengan baseline dan mengukur latency CPU laptop. Jangan menganggap latency GPU Colab sebagai latency laptop. Restart backend dan worker sesudah promosi.

In [ ]:
output = pathlib.Path('/content/ticketops-indobert.zip')
with zipfile.ZipFile(output, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in best.parent.rglob('*'):
        if p.is_file() and 'checkpoints' not in p.parts:
            z.write(p, str(p))
    for p in pathlib.Path('ml/artifacts').glob('evaluation-indobert-*.json'):
        z.write(p, str(p))
    z.write('ml/artifacts/frozen-selection.json')
files.download(str(output))